<a href="https://colab.research.google.com/github/giuliadesantis/Project_IS_Group43/blob/main/STEP8_Cityscapes_Coco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 8 - Anomaly Segmentation EoMT-Coco and EoMT-Cityscapes

# Repository, Drive, Requirements and import SetUp

In [ ]:
!git clone https://github.com/giuliadesantis/Project_IS_Group43.git
%cd Project_IS_Group43/eomt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/Project_IS_Group43')
sys.path.append('/content/Project_IS_Group43/eval')
sys.path.append('/content/Project_IS_Group43/eomt')


In [ ]:
# install requirements
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install ood-metrics lightning
!sed -i '/torch==/d' requirements.txt
!sed -i '/torchvision==/d' requirements.txt
!python3 -m pip install -r requirements.txt

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: Operation cancelled by user
^C


In [ ]:
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

In [ ]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
import warnings
import importlib
import gc

# Setup

In [ ]:
seed_everything(0, verbose=False) # reproducible results

device = 0 # type of torch device
data_path = "/content/drive/MyDrive/CourseProjectAnomaly"  # drive folder of the cityscapes val set to compute the mIoU

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "r") as f:
    config_cs = yaml.safe_load(f)

with open("configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml", "r") as f:
    config_coco = yaml.safe_load(f)

# Load Dataset


In [ ]:
# load cityscapes dataset
data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
)
data.setup()

# Load model

In [ ]:
use_coco = False # boolean set to True if we use EoMT-Coco, False if we use EoMT_Cityscapes

# keep config and bin of the specified model
if use_coco:
    current_config = config_coco
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
    target_img_size = (640, 640)
    num_classes_to_load = 133
else:
    current_config = config_cs
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"
    num_classes_to_load = data.num_classes
    target_img_size = data.img_size

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_to_load,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs,)
    .eval()
    .to(device)
)

# Load Pretrained

In [ ]:
model_kwargs_final = {k: v for k, v in model_kwargs.items()}

if "num_classes" in model_kwargs_final:
  del model_kwargs_final["num_classes"]

name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
is_dinov3 = "dinov3" in name if name else False

if is_dinov3:
    model_kwargs["ckpt_path"] = state_dict_path
    model_kwargs["delta_weights"] = True

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs_final,
    )
    .eval()
    .to(device)
)

if not is_dinov3:
    state_dict = torch.load(
        state_dict_path, map_location=f"cuda:{device}", weights_only=True
    )
    model.load_state_dict(state_dict, strict=False)

# Evaluation Anomaly Metrics

In [ ]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        # ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)


In [ ]:
parser = ArgumentParser()

parser.add_argument(
    "--input",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg", # change the default path based on the dataset folder you want to use
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.webp",
    nargs="+",
    help="A list of space separated input images; "
    "or a single glob pattern such as 'directory/*.jpg'",
)

parser.add_argument('--num-workers', type=int, default=2)
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
parser.add_argument('--device', type=str, default='cuda', help="cpu or cuda, the device used for evaluation")
args = parser.parse_args(args=[])

# initialize anomaly scores lists
anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []
ood_gts_list = []


In [ ]:
# Method to transform the mask prediction of EoMT. We obtain the logits of the pixels.
def get_eomt_logits(model, img_tensor, target_size):
    model.eval()
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img_tensor.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        crops, origins = model.window_imgs_semantic(imgs) # handle crops of EoMT

        mask_logits_per_layer, class_logits_per_layer = model(crops) # masks prediction

        mask_logits = F.interpolate( # Upsampling of masks
            mask_logits_per_layer[-1], target_size, mode="bilinear")

        crop_logits = model.to_per_pixel_logits_semantic( # get the logits
            mask_logits, class_logits_per_layer[-1])

        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes) # reconstruct the image

    return logits

In [ ]:
input_pattern = args.input[0] if isinstance(args.input, list) else args.input

# We create a folder to save logits (this is useful for the next temperature scaling step).
if use_coco:
  #os.makedirs("saved_logits_coco_RoadAnomaly", exist_ok=True) # change the path based on the anomaly dataset
  #os.makedirs("saved_logits_coco_RoadAnomaly21", exist_ok=True)
  #os.makedirs("saved_logits_coco_fs_static", exist_ok=True)
  #os.makedirs("saved_logits_coco_FS_LostFound_full", exist_ok=True)
  os.makedirs("saved_logits_coco_RoadObsticle21", exist_ok=True)
else:
  #os.makedirs("saved_logits_cityscapes_RoadAnomaly", exist_ok=True) # change the path based on the anomaly dataset
  #os.makedirs("saved_logits_cityscapes_RoadAnomaly21", exist_ok=True)
  #os.makedirs("saved_logits_cityscapes_fs_static", exist_ok=True)
  #os.makedirs("saved_logits_cityscapes_FS_LostFound_full", exist_ok=True)
  os.makedirs("saved_logits_cityscapes_RoadObsticle21", exist_ok=True)

for path in glob.glob(os.path.expanduser(str(input_pattern))):

    img_pil = input_transform((Image.open(path).convert('RGB')))
    images = torch.from_numpy(np.array(img_pil)).permute(2, 0, 1)

    logits_list = get_eomt_logits(model, images, target_img_size) # get the logits predictions, it returns a list
    logits = logits_list[0].unsqueeze(0)

    nome_file_logit = os.path.basename(path).replace(".jpg", ".pt").replace(".png", ".pt").replace(".webp", ".pt")

    # we save the logits in the folder we have created
    if use_coco:
      #torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadAnomaly", nome_file_logit))
      #torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadAnomaly21", nome_file_logit))
      #torch.save(logits.cpu(), os.path.join("saved_logits_coco_fs_static", nome_file_logit))
      #torch.save(logits.cpu(), os.path.join("saved_logits_coco_FS_LostFound_full", nome_file_logit))
      torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadObsticle21", nome_file_logit))
    else:
      #torch.save(logits.cpu(), os.path.join("saved_logits_cityscapes_RoadAnomaly", nome_file_logit))
      #torch.save(logits.cpu(), os.path.join("saved_logits_cityscapes_RoadAnomaly21", nome_file_logit))
      #torch.save(logits.cpu(), os.path.join("saved_logits_cityscapes_fs_static", nome_file_logit))
      #torch.save(logits.cpu(), os.path.join("saved_logits_cityscapes_FS_LostFound_full", nome_file_logit))
      torch.save(logits.cpu(), os.path.join("saved_logits_cityscapes_RoadObsticle21", nome_file_logit))

    # Max Logit
    anomaly_result_maxlogit = 1.0 - torch.max(logits.squeeze(0), dim=0)[0].cpu().numpy()

    # MSP
    probs = torch.nn.functional.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs.squeeze(0), dim=0)
    anomaly_result_msp = 1.0 - max_probs.data.cpu().numpy()

    # Max Entropy
    probs = probs.squeeze(0)
    epsilon = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0)
    anomaly_result_maxentropy = entropy.data.cpu().numpy()

    # RbA
    anomaly_result_rba = -torch.sum(torch.tanh(logits.squeeze(0)), dim=0).cpu().numpy()

    # handle images' formats
    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
        # in RoadAnomaly dataset 2 means anomaly. We transform it to standardize wrt other datasets where anomaly is indicated with 1
        ood_gts = np.where((ood_gts==2), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        continue   # if there is no anomaly pixel in the image, skip the image and go to the next
    else:
          ood_gts_list.append(ood_gts)
          anomaly_score_list_maxlogit.append(anomaly_result_maxlogit)
          anomaly_score_list_msp.append(anomaly_result_msp)
          anomaly_score_list_maxentropy.append(anomaly_result_maxentropy)
          anomaly_score_list_rba.append(anomaly_result_rba)

    # to avoid outOfMemory errors
    del logits_list, logits, probs
    del anomaly_result_maxlogit, anomaly_result_msp, anomaly_result_maxentropy, anomaly_result_rba
    del ood_gts, mask, img_pil, images
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# compute the final anomaly detection metrics: AuPRC and FPR95

ood_gts = np.array(ood_gts_list)
anomaly_scores_maxlogit = np.array(anomaly_score_list_maxlogit)
anomaly_scores_msp = np.array(anomaly_score_list_msp)
anomaly_scores_maxentropy = np.array(anomaly_score_list_maxentropy)
anomaly_scores_rba = np.array(anomaly_score_list_rba)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

# Max logit
ood_out = anomaly_scores_maxlogit[ood_mask]
ind_out = anomaly_scores_maxlogit[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AuPRC score maxlogit: {prc_auc*100.0}')
print(f'FPR@TPR95 maxlogit: {fpr*100.0}')


# MSP
ood_out = anomaly_scores_msp[ood_mask]
ind_out = anomaly_scores_msp[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score msp: {prc_auc*100.0}')
print(f'FPR@TPR95 msp: {fpr*100.0}')


# Max entropy
ood_out = anomaly_scores_maxentropy[ood_mask]
ind_out = anomaly_scores_maxentropy[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxentropy: {prc_auc*100.0}')
print(f'FPR@TPR95 maxentropy: {fpr*100.0}')


# RbA
ood_out = anomaly_scores_rba[ood_mask]
ind_out = anomaly_scores_rba[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score rba: {prc_auc*100.0}')
print(f'FPR@TPR95 rba: {fpr*100.0}')

AuPRC score maxlogit: 84.11424132161956
FPR@TPR95 maxlogit: 4.612134459584784
AUPRC score msp: 84.26731731754114
FPR@TPR95 msp: 4.490653029319414
AUPRC score maxentropy: 84.31656884772133
FPR@TPR95 maxentropy: 4.619225628982896
AUPRC score rba: 80.75287064764638
FPR@TPR95 rba: 99.90539587169941


# Temperature scaling


In [ ]:
# we retrieve the saved logits to avoid recomputing them
if use_coco:
  #logits_folder = "saved_logits_coco_RoadAnomaly" # change based on the dataset we use
  #logits_folder = "saved_logits_coco_RoadAnomaly21"
  #logits_folder = "saved_logits_coco_fs_static"
  #logits_folder = "saved_logits_coco_FS_LostFound_full"
  logits_folder = "saved_logits_coco_RoadObsticle21"
else:
  #logits_folder = "saved_logits_cityscapes_RoadAnomaly" # change based on the dataset we use
  #logits_folder = "saved_logits_cityscapes_RoadAnomaly21"
  #logits_folder = "saved_logits_cityscapes_fs_static"
  #logits_folder = "saved_logits_cityscapes_FS_LostFound_full"
  logits_folder = "saved_logits_cityscapes_RoadObsticle21"

#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/labels_masks" # change based on the dataset we use
#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/labels_masks"
#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/labels_masks"
#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/labels_masks"
masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/labels_masks"

# Emprirical grid search over a set of different temperatures
# T=1.0 is the baseline with MSP
temperatures = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 5.0, 10.0, 100.0]

for T in temperatures:
    score_msp_T = []
    gts_T = []

    for logit_path in glob.glob(os.path.join(logits_folder, "*.pt")):

        logits = torch.load(logit_path)
        probs = torch.nn.functional.softmax(logits / T, dim=1)
        max_probs, _ = torch.max(probs.squeeze(0), dim=0)
        anomaly_msp = 1.0 - max_probs.numpy()

        base_name = os.path.basename(logit_path).replace(".pt", "")
        pathGT = os.path.join(masks_folder, f"{base_name}.png")

        if not os.path.exists(pathGT):
            continue

        mask = Image.open(pathGT)
        mask = target_transform(mask)
        ood_gts = np.array(mask)

        if "RoadAnomaly" in masks_folder:
            ood_gts = np.where((ood_gts==2), 1, ood_gts)

        if 1 in np.unique(ood_gts):
            gts_T.append(ood_gts)
            score_msp_T.append(anomaly_msp)

    if len(gts_T) > 0:
        val_gts = np.concatenate(gts_T)
        val_msp = np.concatenate(score_msp_T)

        ood_mask = (val_gts == 1)
        ind_mask = (val_gts == 0)

        val_out_msp = np.concatenate((val_msp[ind_mask], val_msp[ood_mask]))
        val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

        auprc_msp = average_precision_score(val_label, val_out_msp)
        fpr_msp = fpr_at_95_tpr(val_out_msp, val_label)

        print(f"=== Temperature T = {T} ===")
        print(f"AUPRC MSP: {auprc_msp*100:.2f}%  |  FPR95 MSP: {fpr_msp*100:.2f}%\n")
    else:
        print(f"Warning: No anomaly mask for T={T}")

=== Temperature T = 0.01 ===
AUPRC MSP: 64.86%  |  FPR95 MSP: 85.13%

=== Temperature T = 0.05 ===
AUPRC MSP: 83.68%  |  FPR95 MSP: 5.10%

=== Temperature T = 0.1 ===
AUPRC MSP: 84.30%  |  FPR95 MSP: 4.34%

=== Temperature T = 0.2 ===
AUPRC MSP: 84.40%  |  FPR95 MSP: 4.36%

=== Temperature T = 0.5 ===
AUPRC MSP: 84.29%  |  FPR95 MSP: 4.42%

=== Temperature T = 1.0 ===
AUPRC MSP: 84.27%  |  FPR95 MSP: 4.49%

=== Temperature T = 1.5 ===
AUPRC MSP: 84.26%  |  FPR95 MSP: 4.51%

=== Temperature T = 2.0 ===
AUPRC MSP: 84.26%  |  FPR95 MSP: 4.53%

=== Temperature T = 5.0 ===
AUPRC MSP: 84.26%  |  FPR95 MSP: 4.53%

=== Temperature T = 10.0 ===
AUPRC MSP: 84.26%  |  FPR95 MSP: 4.54%

=== Temperature T = 100.0 ===
AUPRC MSP: 84.25%  |  FPR95 MSP: 4.54%

